# Delay Sensitivity Analysis

This notebook checks whether delay findings depend on the definition of a late arrival or on a small number of exceptionally delayed flights. It uses BTS 2025 records for American Airlines departures from Charlotte (CLT).

The first analysis compares arrival-delay thresholds using the same completed-flight denominator. The second refits the RO2 operational regression under several upper-tail training cutoffs. All regression variants are evaluated on the same October–December test set. Trimming is a robustness check, not a recommendation to discard real delay events.

## 1. Load and prepare the data

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

plt.rcParams.update({"figure.figsize": (9, 5), "font.size": 11})
AA_BLUE = "#325AD6"
AA_ORANGE = "#D14000"

working_dir = Path.cwd()
possible_files = [
    working_dir / "Data" / "AA_CLT_2025.csv",
    working_dir.parent / "Data" / "AA_CLT_2025.csv",
]
data_file = next((path for path in possible_files if path.is_file()), None)
if data_file is None:
    raise FileNotFoundError(
        "Can't find Data/AA_CLT_2025.csv. Put the prepared dataset in the project's Data/ folder."
    )

df = pd.read_csv(data_file, low_memory=False)
required_columns = {
    "YEAR", "MONTH", "DAY_OF_WEEK", "FL_DATE", "OP_CARRIER", "ORIGIN",
    "DEST", "CRS_DEP_TIME", "ARR_DELAY", "DEP_DELAY", "TAXI_OUT",
    "CANCELLED", "DIVERTED",
}
missing_columns = sorted(required_columns - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing required BTS columns: {missing_columns}")

df["FL_DATE"] = pd.to_datetime(df["FL_DATE"], format="%m/%d/%Y %I:%M:%S %p")
departures = df.loc[
    df["YEAR"].eq(2025)
    & df["OP_CARRIER"].eq("AA")
    & df["ORIGIN"].eq("CLT")
].copy()
print(f"Loaded {len(df):,} master records; {len(departures):,} are AA departures from CLT.")

## 2. Are delay-rate patterns robust to the threshold?

For each scheduled departure period, calculate the share of completed flights whose arrival delay meets each threshold (10, 15, 20, 30, 45, or 60 minutes). Cancelled and diverted flights are excluded from both numerator and denominator. The consistent denominator makes rates comparable across thresholds.

In [ ]:
period_order = ["Overnight", "Morning", "Afternoon", "Evening"]
departures["SCHED_DEP_HOUR"] = (
    pd.to_numeric(departures["CRS_DEP_TIME"], errors="coerce") // 100
) % 24
departures["DEPARTURE_PERIOD"] = pd.cut(
    departures["SCHED_DEP_HOUR"],
    bins=[-1, 5, 11, 17, 23],
    labels=period_order,
)
departures["COMPLETED"] = (
    departures["CANCELLED"].eq(0)
    & departures["DIVERTED"].eq(0)
    & departures["ARR_DELAY"].notna()
)

thresholds = [10, 15, 20, 30, 45, 60]
threshold_rows = []
for period, group in departures.groupby("DEPARTURE_PERIOD", observed=True):
    completed = group["COMPLETED"]
    denominator = int(completed.sum())
    for threshold in thresholds:
        late_count = int((completed & group["ARR_DELAY"].ge(threshold)).sum())
        threshold_rows.append({
            "Departure period": str(period),
            "Threshold (min)": threshold,
            "Completed flights": denominator,
            "Flights meeting threshold": late_count,
            "Rate (%)": 100 * late_count / denominator if denominator else np.nan,
        })

threshold_results = pd.DataFrame(threshold_rows)
threshold_rates = (
    threshold_results.pivot(
        index="Threshold (min)", columns="Departure period", values="Rate (%)"
    )
    .reindex(index=thresholds, columns=period_order)
)
display(threshold_rates.round(2))

fig, ax = plt.subplots(figsize=(10, 5))
for period in period_order:
    ax.plot(
        threshold_rates.index,
        threshold_rates[period],
        marker="o",
        linewidth=2,
        label=period,
    )
ax.set(
    title="Arrival-delay rates across thresholds by scheduled departure period",
    xlabel="Arrival-delay threshold (minutes)",
    ylabel="Share of completed flights meeting threshold (%)",
    xticks=thresholds,
)
ax.legend(title="Departure period", frameon=False)
ax.grid(axis="y", alpha=.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 3. Are regression results sensitive to extreme training delays?

Fit the same operational model as RO2 using January–September flights, and evaluate each fit on the unchanged October–December test period. Destination categories are grouped using training data only; the low-volume destination rule matches RO2 (fewer than 100 training flights become `OTHER`). Each percentile cutoff is also calculated from training delays only. Ties at a cutoff can make the actual excluded share differ slightly from the nominal percentile.

In [ ]:
completed = departures.loc[
    departures["CANCELLED"].eq(0)
    & departures["DIVERTED"].eq(0)
    & departures["ARR_DELAY"].notna()
].copy()

completed["DEP_HOUR"] = pd.to_numeric(completed["CRS_DEP_TIME"], errors="coerce") // 100
completed.loc[completed["DEP_HOUR"].eq(24), "DEP_HOUR"] = 0
completed["DEP_PERIOD"] = pd.cut(
    completed["DEP_HOUR"],
    bins=[-1, 5, 11, 17, 23],
    labels=["Overnight", "Morning", "Afternoon", "Evening"],
).astype("object")
completed["MONTH_SIN"] = np.sin(2 * np.pi * completed["MONTH"] / 12)
completed["MONTH_COS"] = np.cos(2 * np.pi * completed["MONTH"] / 12)
completed["WEEKDAY"] = completed["DAY_OF_WEEK"].astype("string")

categorical = ["DEST", "DEP_PERIOD", "WEEKDAY"]
scheduled_numeric = ["MONTH_SIN", "MONTH_COS"]
operational_numeric = ["DEP_DELAY", "TAXI_OUT"]
model_features = categorical + scheduled_numeric + operational_numeric
model_df = completed.dropna(subset=["ARR_DELAY"] + model_features).copy()

train = model_df.loc[model_df["MONTH"] <= 9].copy()
test = model_df.loc[model_df["MONTH"] >= 10].copy()
if train.empty or test.empty:
    raise ValueError("Both Jan–Sep training and Oct–Dec test records are required.")
if train["FL_DATE"].max() >= test["FL_DATE"].min():
    raise ValueError("Training dates must precede all test dates.")

common_destinations = train["DEST"].value_counts()
common_destinations = common_destinations[common_destinations >= 100].index
train["DEST"] = train["DEST"].where(train["DEST"].isin(common_destinations), "OTHER")
test["DEST"] = test["DEST"].where(test["DEST"].isin(common_destinations), "OTHER")

all_features = categorical + scheduled_numeric + operational_numeric
y_train = train["ARR_DELAY"]
y_test = test["ARR_DELAY"]
print(f"Training flights: {len(train):,} (Jan–Sep)")
print(f"Test flights: {len(test):,} (Oct–Dec; held fixed for all fits)")

def regression_model(numeric_features):
    preprocessor = ColumnTransformer(
        transformers=[
            ("categories", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical),
            ("minutes", "passthrough", numeric_features),
        ]
    )
    return Pipeline([
        ("prepare", preprocessor),
        ("regression", LinearRegression()),
    ])

def score_predictions(y_true, y_pred):
    return {
        "MAE (min)": mean_absolute_error(y_true, y_pred),
        "RMSE (min)": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R²": r2_score(y_true, y_pred),
    }

cutoff_percentiles = [95, 97.5, 99, 99.5, 100]
sensitivity_rows = []
for percentile in cutoff_percentiles:
    upper_bound = y_train.quantile(percentile / 100)
    sensitivity_train = train.loc[train["ARR_DELAY"] <= upper_bound]
    model = regression_model(scheduled_numeric + operational_numeric)
    model.fit(sensitivity_train[all_features], sensitivity_train["ARR_DELAY"])
    predictions = model.predict(test[all_features])
    coefficients = pd.Series(
        model.named_steps["regression"].coef_,
        index=model.named_steps["prepare"].get_feature_names_out(),
    )
    sensitivity_rows.append({
        "Training cutoff": f"p{percentile:g}",
        "Upper bound (min)": upper_bound,
        "Training flights retained": len(sensitivity_train),
        "Training flights excluded": len(train) - len(sensitivity_train),
        "Excluded (%)": 100 * (len(train) - len(sensitivity_train)) / len(train),
        **score_predictions(y_test, predictions),
        "DEP_DELAY coefficient": coefficients["minutes__DEP_DELAY"],
        "TAXI_OUT coefficient": coefficients["minutes__TAXI_OUT"],
    })

sensitivity_results = pd.DataFrame(sensitivity_rows).set_index("Training cutoff")
display(sensitivity_results.round(3))

## Interpretation

- If the departure-period ranking is similar across thresholds, that descriptive pattern is less dependent on choosing a particular definition of “late.” The rates still describe association, not the cause of delays.
- Compare MAE, RMSE, and R² across regression cutoffs to see how trimming training outliers changes predictions for the same later flights. A model trained after trimming is not automatically better for operational planning because extreme delays are real outcomes.
- The `DEP_DELAY` and `TAXI_OUT` coefficients are conditional associations. In particular, departure delay is closely related to arrival delay, so its coefficient is not a causal estimate.